In [1]:
%env SEC_USER_AGENT=Charles-IESEG-Research/1.0 tonmail@example.com

env: SEC_USER_AGENT=Charles-IESEG-Research/1.0 tonmail@example.com


In [2]:
%run /content/option3_run_audited_workbook_V2.py

/content/option3_run_audited_workbook_V2.py:83: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  firm_master["cik"] = firm_master["cik"].fillna(firm_master["cik_secmap"])


Yahoo extraction:   0%|          | 0/200 [00:00<?, ?it/s]

ERROR:yfinance:$UBER: possibly delisted; no price data found  (1d 2018-01-01 -> 2019-01-10) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1547096400")
ERROR:yfinance:$UBER: possibly delisted; no price data found  (1d 2018-01-01 -> 2019-01-10) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1547096400")
ERROR:yfinance:$ABNB: possibly delisted; no price data found  (1d 2018-01-01 -> 2019-01-10) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1547096400")
ERROR:yfinance:$ABNB: possibly delisted; no price data found  (1d 2018-01-01 -> 2019-01-10) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1547096400")
ERROR:yfinance:$ABNB: possibly delisted; no price data found  (1d 2019-01-01 -> 2020-01-10) (Yahoo error = "Data doesn't exist for startDate = 1546318800, endDate = 1578632400")
ERROR:yfinance:$ABNB: possibly delisted; no price data found  (1d 2019-01-01 -> 2020-01-10) (Yahoo error = "Da

SEC extraction:   0%|          | 0/90 [00:00<?, ?it/s]

/content/option3_run_audited_workbook_V2.py:749: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  panel["sales_growth"] = panel.groupby("ticker")["revenue"].pct_change()
/content/option3_run_audited_workbook_V2.py:750: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  panel["asset_growth"] = panel.groupby("ticker")["total_assets"].pct_change()


Done V2.
Outputs written to: /content/option3_outputs
Final workbook: /content/option3_outputs/Option3_FullyAutomated_Master_POPULATED_V2.xlsx
SEC rows: 671 | filings index rows: 737
submission_ok      0.988889
companyfacts_ok    0.966667
dtype: float64
US rows with SEC overrides expected: 661


In [3]:
import pandas as pd

panel = pd.read_csv('/content/option3_outputs/06_panel_annual_clean.csv')
secdiag = pd.read_csv('/content/option3_outputs/07_sec_diagnostics.csv')

print(panel['source_primary'].value_counts(dropna=False))
print(panel[['country','macro_zone']].drop_duplicates().sort_values(['macro_zone','country']))
print(secdiag.head())
print(secdiag[['submission_ok','companyfacts_ok']].mean(numeric_only=True))

source_primary
Yahoo only              939
Yahoo + SEC override    661
Name: count, dtype: int64
             country      macro_zone
288        Australia       Australia
208           Canada          Canada
264           France       Euro Area
240          Germany       Euro Area
736          Ireland       Euro Area
216      Netherlands       Euro Area
0              Japan           Japan
1584     New Zealand     New Zealand
176      Switzerland     Switzerland
168    United States              US
160   United Kingdom  United Kingdom
  ticker firm_id    cik10  submission_ok  companyfacts_ok  rows_extracted  \
0   AAPL    F001   320193           True             True               8   
1   MSFT    F002   789019           True             True               8   
2   NVDA    F003  1045810           True             True               8   
3   AMZN    F004  1018724           True             True               8   
4  GOOGL    F005  1652044           True             True               8 

In [4]:
panel['quality_flag'].value_counts(dropna=False)

,count
quality_flag,
GOOD,992
DROP,473
REVIEW,135


In [5]:
panel.groupby('fiscal_year')['quality_flag'].value_counts().unstack(fill_value=0)

quality_flag,DROP,GOOD,REVIEW
fiscal_year,,,
2018,116,50,34
2019,116,51,33
2020,116,50,34
2021,109,60,31
2022,2,196,2
2023,2,198,0
2024,2,198,0
2025,10,189,1


In [6]:
panel.groupby('country')['quality_flag'].value_counts().unstack(fill_value=0)

quality_flag,DROP,GOOD,REVIEW
country,,,
Australia,36,36,0
Canada,68,52,0
France,56,56,0
Germany,60,60,0
Ireland,4,9,3
Japan,80,80,0
Netherlands,44,43,1
New Zealand,4,4,0
Switzerland,44,44,0


In [7]:
key_cols = [
    'total_assets','total_debt','revenue','ebitda','ebit',
    'interest_expense','net_income','market_cap_fye',
    'avg_policy_rate','policy_rate_fye'
]

panel[key_cols].notna().mean().sort_values()

,0
ebitda,0.490000
interest_expense,0.662500
ebit,0.667500
total_debt,0.685625
net_income,0.700625
revenue,0.700625
total_assets,0.705000
market_cap_fye,0.983750
avg_policy_rate,0.995000
policy_rate_fye,0.995000


In [8]:
panel['source_primary'].value_counts(dropna=False)


,count
source_primary,
Yahoo only,939
Yahoo + SEC override,661


In [9]:
import pandas as pd

panel = pd.read_csv('/content/option3_outputs/06_panel_annual_clean.csv')

# Base 2022-2025, lignes utilisables
base = panel[(panel['fiscal_year'].between(2022, 2025)) & (panel['quality_flag'] == 'GOOD')].copy()

# Combinaisons utiles
checks = {
    'roa_ebit_assets': ['ebit', 'total_assets'],
    'roa_ni_assets': ['net_income', 'total_assets'],
    'leverage_debt_assets': ['total_debt', 'total_assets'],
    'interest_coverage': ['ebit', 'interest_expense'],
    'debt_ebitda': ['total_debt', 'ebitda'],
    'ebitda_margin': ['ebitda', 'revenue'],
    'market_based': ['market_cap_fye', 'avg_policy_rate', 'policy_rate_fye']
}

for name, cols in checks.items():
    rate = base[cols].notna().all(axis=1).mean()
    print(f"{name}: {rate:.3%}")

roa_ebit_assets: 100.000%
roa_ni_assets: 100.000%
leverage_debt_assets: 96.671%
interest_coverage: 99.104%
debt_ebitda: 95.775%
ebitda_margin: 99.104%
market_based: 99.488%


In [10]:
key_cols = [
    'total_assets','total_debt','revenue','ebitda','ebit',
    'interest_expense','net_income'
]

for col in key_cols:
    print(f"\n===== {col} =====")
    print(panel.groupby('fiscal_year')[col].apply(lambda s: s.notna().mean()))
    print(panel.groupby('country')[col].apply(lambda s: s.notna().mean()).sort_values())
    print(panel.groupby('source_primary')[col].apply(lambda s: s.notna().mean()))


===== total_assets =====
fiscal_year
2018    0.420
2019    0.420
2020    0.420
2021    0.465
2022    0.990
2023    0.990
2024    0.990
2025    0.945
Name: total_assets, dtype: float64
country
Canada            0.433333
Australia         0.500000
France            0.500000
Germany           0.500000
Japan             0.500000
Netherlands       0.500000
New Zealand       0.500000
Switzerland       0.500000
United Kingdom    0.516667
Ireland           0.750000
United States     0.974138
Name: total_assets, dtype: float64
source_primary
Yahoo + SEC override    0.996974
Yahoo only              0.499468
Name: total_assets, dtype: float64

===== total_debt =====
fiscal_year
2018    0.395
2019    0.395
2020    0.405
2021    0.500
2022    0.960
2023    0.955
2024    0.955
2025    0.920
Name: total_debt, dtype: float64
country
Japan             0.381250
Canada            0.450000
New Zealand       0.500000
France            0.500000
Germany           0.508333
Netherlands       0.511364
Switzerl

In [11]:
print(panel['quality_flag'].value_counts(dropna=False))

suspect = panel[panel['quality_flag'].isin(['DROP', 'REVIEW'])].copy()

cols_to_look = [
    'ticker','country','fiscal_year','source_primary','quality_flag',
    'total_assets','total_debt','revenue','ebitda','ebit',
    'interest_expense','net_income','market_cap_fye'
]

print(suspect[cols_to_look].head(50).to_string())

quality_flag
GOOD      992
DROP      473
REVIEW    135
Name: count, dtype: int64
    ticker country  fiscal_year source_primary quality_flag  total_assets  total_debt  revenue  ebitda  ebit  interest_expense  net_income  market_cap_fye
0   4063.T   Japan         2018     Yahoo only         DROP           NaN         NaN      NaN     NaN   NaN               NaN         NaN    7.465154e+11
1   4063.T   Japan         2019     Yahoo only         DROP           NaN         NaN      NaN     NaN   NaN               NaN         NaN    9.714720e+11
2   4063.T   Japan         2020     Yahoo only         DROP           NaN         NaN      NaN     NaN   NaN               NaN         NaN    1.579577e+12
3   4063.T   Japan         2021     Yahoo only         DROP           NaN         NaN      NaN     NaN   NaN               NaN         NaN    1.648634e+12
8   4519.T   Japan         2018     Yahoo only         DROP           NaN         NaN      NaN     NaN   NaN               NaN         NaN    1.

In [12]:
base = panel[(panel['fiscal_year'].between(2022, 2025)) & (panel['quality_flag'] == 'GOOD')].copy()

firm_years = base.groupby('ticker').size().sort_values()
print(firm_years.describe())

print("\nFirmes avec 4 années GOOD sur 2022-2025 :")
print((firm_years == 4).sum())

print("\nFirmes avec au moins 3 années GOOD :")
print((firm_years >= 3).sum())

count    198.000000
mean       3.944444
std        0.229642
min        3.000000
25%        4.000000
50%        4.000000
75%        4.000000
max        4.000000
dtype: float64

Firmes avec 4 années GOOD sur 2022-2025 :
187

Firmes avec au moins 3 années GOOD :
198


In [13]:
import os
from glob import glob

files = glob('/content/option3_outputs/*.xlsx')
print(files)

['/content/option3_outputs/Option3_FullyAutomated_Master_POPULATED_V2.xlsx']


In [15]:
from glob import glob
from google.colab import files

xlsx_files = glob('/content/option3_outputs/*.xlsx')

if len(xlsx_files) == 0:
    print("Aucun fichier Excel trouvé dans /content/option3_outputs/")
else:
    print("Téléchargement de :", xlsx_files[0])
    files.download(xlsx_files[0])

Téléchargement de : /content/option3_outputs/Option3_FullyAutomated_Master_POPULATED_V2.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
from google.colab import files

uploaded = files.upload()
print(uploaded.keys())

Saving Option3_FullyAutomated_Master_POPULATED_V2.xlsx to Option3_FullyAutomated_Master_POPULATED_V2.xlsx
dict_keys(['Option3_FullyAutomated_Master_POPULATED_V2.xlsx'])


In [18]:
import os

print(os.listdir('/content'))

['.config', 'option3_run_audited_workbook_V2.py', 'option3_outputs', 'Option3_Firm_Master_200_FILLED.xlsx', 'Option3_FullyAutomated_Master_POPULATED_V2.xlsx', 'sample_data']


In [19]:
import os
import glob
import pandas as pd

excel_files = glob.glob('/content/*.xlsx')

print("Fichiers Excel trouvés :", excel_files)

if len(excel_files) == 0:
    raise FileNotFoundError("Aucun fichier Excel trouvé dans /content")

file_path = excel_files[0]
print("Fichier utilisé :", file_path)

panel = pd.read_excel(file_path, sheet_name="panel_annual_clean")
print("Shape :", panel.shape)
panel.head()

Fichiers Excel trouvés : ['/content/Option3_Firm_Master_200_FILLED.xlsx', '/content/Option3_FullyAutomated_Master_POPULATED_V2.xlsx']
Fichier utilisé : /content/Option3_Firm_Master_200_FILLED.xlsx
Shape : (0, 45)


,firm_id,ticker,company_name,country,region,sector,reporting_currency,accounting_standard,fiscal_year,filing_date,...,tangibility,liquidity,size_ln_assets,sales_growth,asset_growth,interest_coverage,tobins_q_proxy,data_quality_score,quality_flag,notes


In [21]:
import os
import glob
import pandas as pd
import numpy as np

# ========= 1) FIND FILE AUTOMATICALLY =========
# The original code was trying to load panel from an Excel file that might be empty or not contain the right data.
# Instead, we should load the panel from the already processed CSV output.
# excel_files = glob.glob('/content/*.xlsx')
# if len(excel_files) == 0:
#     raise FileNotFoundError("Aucun fichier Excel trouvé dans /content")
# file_path = excel_files[0]
# print("Using file:", file_path)

# ========= 2) LOAD =========
# panel = pd.read_excel(file_path, sheet_name="panel_annual_clean") # Original line
panel = pd.read_csv('/content/option3_outputs/06_panel_annual_clean.csv') # Corrected line
print("Panel loaded from: /content/option3_outputs/06_panel_annual_clean.csv")

# ========= 3) MAIN ESTIMATION SAMPLE =========
base = panel[
    (panel["fiscal_year"].between(2022, 2025)) &
    (panel["quality_flag"] == "GOOD")
].copy()

# ========= 4) CREATE CORE VARIABLES =========
base["roa_ebit"] = base["ebit"] / base["total_assets"]
base["roa_ni"] = base["net_income"] / base["total_assets"]
base["leverage"] = base["total_debt"] / base["total_assets"]
base["interest_coverage_calc"] = base["ebit"] / base["interest_expense"]
base["debt_ebitda"] = base["total_debt"] / base["ebitda"]
base["ebitda_margin"] = base["ebitda"] / base["revenue"]
base["log_assets"] = np.log(base["total_assets"])

base = base.sort_values(["macro_zone", "fiscal_year", "firm_id"]).copy()
base["policy_rate_change"] = base.groupby("macro_zone")["policy_rate_fye"].diff()

base = base.sort_values(["firm_id", "fiscal_year"]).copy()
base["leverage_lag"] = base.groupby("firm_id")["leverage"].shift(1)

base["rate_x_leverage_lag"] = base["policy_rate_fye"] * base["leverage_lag"]

ratio_cols = [
    "roa_ebit", "roa_ni", "leverage",
    "interest_coverage_calc", "debt_ebitda",
    "ebitda_margin", "log_assets",
    "policy_rate_fye", "policy_rate_change",
    "leverage_lag", "rate_x_leverage_lag"
]

desc = base[ratio_cols].describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]).T
desc["missing_pct"] = base[ratio_cols].isna().mean().values

def winsorize_series(s, lower=0.01, upper=0.99):
    if s.dropna().empty:
        return s
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lower=lo, upper=hi)

winsor_cols = [
    "roa_ebit", "roa_ni", "leverage",
    "interest_coverage_calc", "debt_ebitda",
    "ebitda_margin", "log_assets"
]

base_w = base.copy()
for col in winsor_cols:
    base_w[col] = winsorize_series(base_w[col], 0.01, 0.99)

checks = pd.DataFrame({
    "variable": ratio_cols,
    "missing_pct": [base[c].isna().mean() for c in ratio_cols],
    "negative_pct": [(base[c] < 0).mean() for c in ratio_cols],
    "inf_pct": [np.isinf(base[c]).mean() for c in ratio_cols]
})

sample_info = pd.DataFrame({
    "metric": [
        "rows_main_sample",
        "firms_main_sample",
        "mean_years_per_firm",
        "firms_with_4_years",
        "firms_with_at_least_3_years"
    ],
    "value": [
        len(base),
        base["firm_id"].nunique(),
        base.groupby("firm_id").size().mean(),
        (base.groupby("firm_id").size() == 4).sum(),
        (base.groupby("firm_id").size() >= 3).sum()
    ]
})

output_path = "/content/analysis_ready_panel_2022_2025.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    base.to_excel(writer, sheet_name="analysis_base_raw", index=False)
    base_w.to_excel(writer, sheet_name="analysis_base_winsor", index=False)
    desc.to_excel(writer, sheet_name="diagnostics_desc")
    checks.to_excel(writer, sheet_name="diagnostics_checks", index=False)
    sample_info.to_excel(writer, sheet_name="sample_info", index=False)

print("Saved:", output_path)

Panel loaded from: /content/option3_outputs/06_panel_annual_clean.csv
Saved: /content/analysis_ready_panel_2022_2025.xlsx


In [22]:
from google.colab import files
files.download('/content/analysis_ready_panel_2022_2025.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
from google.colab import files

uploaded = files.upload()

Saving analysis_ready_panel_2022_2025.xlsx to analysis_ready_panel_2022_2025.xlsx


In [4]:
import pandas as pd
import numpy as np

file_path = "/content/analysis_ready_panel_2022_2025.xlsx"   # adapte si besoin
df = pd.read_excel(file_path, sheet_name="analysis_base_winsor")

print(df.shape)
print(df.columns.tolist())
df.head()

(781, 56)
['firm_id', 'ticker', 'company_name', 'country', 'region', 'sector', 'macro_zone', 'reporting_currency', 'accounting_standard', 'fiscal_year', 'filing_date', 'source_primary', 'total_assets', 'total_debt', 'short_term_debt', 'long_term_debt', 'cash_and_equivalents', 'total_equity', 'revenue', 'ebitda', 'ebit', 'interest_expense', 'net_income', 'ppe_net', 'capex', 'operating_cash_flow', 'depreciation_amortization', 'shares_outstanding', 'market_cap_fye', 'avg_policy_rate', 'policy_rate_fye', 'book_leverage', 'market_leverage', 'net_leverage', 'roa', 'roe', 'tangibility', 'liquidity', 'size_ln_assets', 'sales_growth', 'asset_growth', 'interest_coverage', 'tobins_q_proxy', 'data_quality_score', 'quality_flag', 'notes', 'roa_ebit', 'roa_ni', 'leverage', 'interest_coverage_calc', 'debt_ebitda', 'ebitda_margin', 'log_assets', 'policy_rate_change', 'leverage_lag', 'rate_x_leverage_lag']


,firm_id,ticker,company_name,country,region,sector,macro_zone,reporting_currency,accounting_standard,fiscal_year,...,roa_ebit,roa_ni,leverage,interest_coverage_calc,debt_ebitda,ebitda_margin,log_assets,policy_rate_change,leverage_lag,rate_x_leverage_lag
0,F001,AAPL,Apple Inc.,United States,US,Technology,US,USD,US GAAP,2022,...,0.323701,0.276664,0.317002,40.749574,0.856620,0.331047,26.589040,NaN,NaN,NaN
1,F001,AAPL,Apple Inc.,United States,US,Technology,US,USD,US GAAP,2023,...,0.323701,0.275098,0.302261,29.062039,0.847020,0.328267,26.588552,1.23,0.317002,1.689620
2,F001,AAPL,Apple Inc.,United States,US,Technology,US,USD,US GAAP,2024,...,0.323701,0.256825,0.266702,NaN,0.722860,0.344371,26.623108,-0.85,0.302261,1.354128
3,F001,AAPL,Apple Inc.,United States,US,Technology,US,USD,US GAAP,2025,...,0.323701,0.276664,0.250099,NaN,0.630620,0.347817,26.623108,-0.76,0.266702,0.992133
4,F002,MSFT,Microsoft Corporation,United States,US,Technology,US,USD,US GAAP,2022,...,0.145157,0.121371,0.152152,20.439599,0.553786,0.700899,26.622725,0.00,NaN,NaN


In [5]:
# 1) table unique des taux par zone et année
zone_year = (
    df[["macro_zone", "fiscal_year", "policy_rate_fye"]]
    .drop_duplicates()
    .sort_values(["macro_zone", "fiscal_year"])
    .copy()
)

# 2) variation correcte du taux par zone
zone_year["policy_rate_change_correct"] = (
    zone_year.groupby("macro_zone")["policy_rate_fye"].diff()
)

# 3) on remerge dans la base firme-année
df = df.drop(columns=["policy_rate_change"], errors="ignore").merge(
    zone_year[["macro_zone", "fiscal_year", "policy_rate_change_correct"]],
    on=["macro_zone", "fiscal_year"],
    how="left"
)

# 4) on renomme proprement
df = df.rename(columns={"policy_rate_change_correct": "policy_rate_change"})

# Vérification
print(df[["macro_zone", "fiscal_year", "policy_rate_fye", "policy_rate_change"]]
      .drop_duplicates()
      .sort_values(["macro_zone", "fiscal_year"])
      .head(20))

      macro_zone  fiscal_year  policy_rate_fye  policy_rate_change
741    Australia         2022         3.050000                 NaN
742    Australia         2023         4.350000            1.300000
743    Australia         2024         4.350000            0.000000
744    Australia         2025         3.600000           -0.750000
494       Canada         2022         4.124500                 NaN
495       Canada         2023         5.015479            0.890979
496       Canada         2024         3.468520           -1.546959
497       Canada         2025         2.251419           -1.217101
314    Euro Area         2022         1.503965                 NaN
315    Euro Area         2023         3.903442            2.399477
316    Euro Area         2024         3.102686           -0.800756
317    Euro Area         2025         1.929287           -1.173400
358        Japan         2022        -0.070000                 NaN
359        Japan         2023        -0.012000            0.05

In [6]:
cols_to_drop = [
    "roa_ni",
    "leverage",
    "interest_coverage_calc",
    "log_assets"
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print(df.shape)

(781, 52)


In [7]:
df.loc[df["ebitda"] <= 0, "debt_ebitda"] = np.nan
df.loc[df["interest_expense"] <= 0, "interest_coverage"] = np.nan

print("Rows with debt_ebitda missing:", df["debt_ebitda"].isna().sum())
print("Rows with interest_coverage missing:", df["interest_coverage"].isna().sum())

Rows with debt_ebitda missing: 39
Rows with interest_coverage missing: 7


In [8]:
vars_check = [
    "roa",
    "roa_ebit",
    "book_leverage",
    "interest_coverage",
    "debt_ebitda",
    "ebitda_margin",
    "size_ln_assets",
    "policy_rate_fye",
    "policy_rate_change",
    "leverage_lag",
    "rate_x_leverage_lag"
]

desc = df[vars_check].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
desc["missing_pct"] = df[vars_check].isna().mean()

print(desc)

                     count       mean         std         min         1%  \
roa                  781.0   0.083080    0.070410   -0.321651  -0.075651   
roa_ebit             781.0   0.112353    0.071872   -0.057531  -0.057151   
book_leverage        755.0   0.275148    0.155007    0.000000   0.007499   
interest_coverage    774.0  77.995235  709.751186 -597.391304  -3.974793   
debt_ebitda          742.0   2.208239    1.991132    0.011371   0.033227   
ebitda_margin        774.0   0.294960    0.211198    0.019581   0.020176   
size_ln_assets       781.0  25.204966    1.780191   21.501686  22.039428   
policy_rate_fye      777.0   3.335225    1.580345   -0.070000  -0.070000   
policy_rate_change   582.0   0.014706    1.129953   -1.546959  -1.546959   
leverage_lag         564.0   0.276280    0.154346    0.000000   0.007499   
rate_x_leverage_lag  561.0   1.037559    0.776859   -0.004697  -0.001823   

                            5%        25%        50%        75%         95%  \
roa     

In [9]:
df["rate_change_x_leverage_lag"] = df["policy_rate_change"] * df["leverage_lag"]

print(df[[
    "policy_rate_change",
    "leverage_lag",
    "rate_change_x_leverage_lag"
]].head(10))

   policy_rate_change  leverage_lag  rate_change_x_leverage_lag
0                 NaN           NaN                         NaN
1                1.23      0.317002                    0.389912
2               -0.85      0.302261                   -0.256922
3               -0.76      0.266702                   -0.202694
4                 NaN           NaN                         NaN
5                1.23      0.152152                    0.187147
6               -0.85      0.152152                   -0.129329
7               -0.76      0.100009                   -0.076007
8                 NaN           NaN                         NaN
9                1.23      0.247720                    0.304695


In [10]:
final_cols = [
    "firm_id",
    "ticker",
    "company_name",
    "country",
    "macro_zone",
    "sector",
    "industry",
    "fiscal_year",

    # dependent variables
    "roa",
    "roa_ebit",
    "interest_coverage",
    "debt_ebitda",
    "ebitda_margin",
    "book_leverage",

    # main explanatory variables
    "policy_rate_fye",
    "policy_rate_change",
    "leverage_lag",
    "rate_x_leverage_lag",
    "rate_change_x_leverage_lag",

    # controls
    "size_ln_assets",
    "revenue",
    "ebit",
    "ebitda",
    "interest_expense",
    "net_income",
    "total_assets",
    "total_debt",

    # optional quality info
    "source_primary",
    "data_quality_score",
    "notes"
]

final_df = df[[c for c in final_cols if c in df.columns]].copy()

print(final_df.shape)
final_df.head()

(781, 29)


,firm_id,ticker,company_name,country,macro_zone,sector,fiscal_year,roa,roa_ebit,interest_coverage,...,revenue,ebit,ebitda,interest_expense,net_income,total_assets,total_debt,source_primary,data_quality_score,notes
0,F001,AAPL,Apple Inc.,United States,US,Technology,2022,0.282924,0.323701,40.749574,...,3.943280e+11,1.194370e+11,1.305410e+11,2.931000e+09,9.980300e+10,3.527550e+11,1.118240e+11,Yahoo + SEC override,1.000,NaN
1,F001,AAPL,Apple Inc.,United States,US,Technology,2023,0.275098,0.323701,29.062039,...,3.832850e+11,1.143010e+11,1.258200e+11,3.933000e+09,9.699500e+10,3.525830e+11,1.065720e+11,Yahoo + SEC override,1.000,NaN
2,F001,AAPL,Apple Inc.,United States,US,Technology,2024,0.256825,0.323701,NaN,...,3.910350e+11,1.232160e+11,1.346610e+11,NaN,9.373600e+10,3.649800e+11,9.734100e+10,Yahoo + SEC override,0.875,NaN
3,F001,AAPL,Apple Inc.,United States,US,Technology,2025,0.306894,0.323701,NaN,...,4.161610e+11,1.330500e+11,1.447480e+11,NaN,1.120100e+11,3.649800e+11,9.128100e+10,Yahoo + SEC override,0.875,NaN
4,F002,MSFT,Microsoft Corporation,United States,US,Technology,2022,0.121371,0.145157,20.439599,...,1.430150e+11,5.295900e+10,1.002390e+11,2.591000e+09,4.428100e+10,3.648400e+11,5.551100e+10,Yahoo + SEC override,1.000,NaN


In [11]:
output_path = "/content/final_regression_dataset_v1.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    final_df.to_excel(writer, sheet_name="regression_base", index=False)
    desc.to_excel(writer, sheet_name="diagnostics_desc")
    zone_year.to_excel(writer, sheet_name="macro_zone_rates", index=False)

print("Saved:", output_path)

Saved: /content/final_regression_dataset_v1.xlsx


In [12]:
from google.colab import files
files.download("/content/final_regression_dataset_v1.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>